# Texas Workers' Compensation Claims — Trend Analysis

This notebook analyzes medical billing trends from the **Texas Division of Workers' Compensation** open data, covering **Professional (SV1)**, **Institutional (SV2)**, and **Pharmacy (SV4)** claims from 2010–2026.

**Data source:** [Texas Open Data Portal — Medical Billing](https://data.texas.gov/browse?q=medical+billing&sortBy=relevance)

### Sections
1. **Setup & Data Loading** — Connect to DuckDB and build unified views
2. **Overall Cost Trends** — Charges vs. payments over time by claim type
3. **Professional Claims Analysis** — Top procedures, diagnosis patterns, reimbursement rates
4. **Pharmacy Claims Analysis** — Drug prescribing trends, opioid monitoring, cost drivers
5. **Institutional Claims Analysis** — Facility billing patterns and high-cost episodes
6. **Reimbursement Analysis** — Paid-to-billed ratios and payment lag trends

In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

con = duckdb.connect('../tx_workers_comp.db', read_only=True)

# Unified header view — select shared columns, NULL for missing ones
all_headers = con.sql("""
    SELECT bill_id, 'professional' as claim_category, bill_type,
        CAST(date_of_bill AS DATE) AS bill_date,
        CAST(service_bill_from_date AS DATE) AS service_from,
        CAST(service_bill_to_date AS DATE) AS service_to,
        CAST(total_charge_per_bill AS DOUBLE) AS charge,
        CAST(total_amount_paid_per_bill AS DOUBLE) AS paid,
        CAST(date_insurer_received_bill AS DATE) AS insurer_received,
        CAST(date_insurer_paid_bill AS DATE) AS insurer_paid,
        CAST(employee_date_of_injury AS DATE) AS injury_date,
        employee_gender_code AS gender,
        first_icd_9cm_or_icd_10cm AS dx1,
        second_icd_9cm_or_icd_10cm AS dx2
    FROM raw.professional_header_current
    UNION ALL
    SELECT bill_id, 'professional', bill_type,
        CAST(date_of_bill AS DATE), CAST(service_bill_from_date AS DATE),
        CAST(service_bill_to_date AS DATE), CAST(total_charge_per_bill AS DOUBLE),
        CAST(total_amount_paid_per_bill AS DOUBLE), CAST(date_insurer_received_bill AS DATE),
        CAST(date_insurer_paid_bill AS DATE), CAST(employee_date_of_injury AS DATE),
        employee_gender_code, first_icd_9cm_or_icd_10cm, second_icd_9cm_or_icd_10cm
    FROM raw.professional_header_historical
    UNION ALL
    SELECT bill_id, 'pharmacy', bill_type,
        CAST(date_of_bill AS DATE), CAST(service_bill_from_date AS DATE),
        CAST(service_bill_to_date AS DATE), CAST(total_charge_per_bill AS DOUBLE),
        CAST(total_amount_paid_per_bill AS DOUBLE), CAST(date_insurer_received_bill AS DATE),
        CAST(date_insurer_paid_bill AS DATE), CAST(employee_date_of_injury AS DATE),
        employee_gender_code, NULL AS dx1, NULL AS dx2
    FROM raw.pharmacy_header_current
    UNION ALL
    SELECT bill_id, 'pharmacy', bill_type,
        CAST(date_of_bill AS DATE), CAST(service_bill_from_date AS DATE),
        CAST(service_bill_to_date AS DATE), CAST(total_charge_per_bill AS DOUBLE),
        CAST(total_amount_paid_per_bill AS DOUBLE), CAST(date_insurer_received_bill AS DATE),
        CAST(date_insurer_paid_bill AS DATE), CAST(employee_date_of_injury AS DATE),
        employee_gender_code, NULL, NULL
    FROM raw.pharmacy_header_historical
    UNION ALL
    SELECT bill_id, 'institutional', bill_type,
        CAST(date_of_bill AS DATE), CAST(service_bill_from_date AS DATE),
        CAST(service_bill_to_date AS DATE), CAST(total_charge_per_bill AS DOUBLE),
        CAST(total_amount_paid_per_bill AS DOUBLE), CAST(date_insurer_received_bill AS DATE),
        CAST(date_insurer_paid_bill AS DATE), CAST(employee_date_of_injury AS DATE),
        employee_gender_code, first_icd_9cm_or_icd_10cm, second_icd_9cm_or_icd_10cm
    FROM raw.institutional_header_current
    UNION ALL
    SELECT bill_id, 'institutional', bill_type,
        CAST(date_of_bill AS DATE), CAST(service_bill_from_date AS DATE),
        CAST(service_bill_to_date AS DATE), CAST(total_charge_per_bill AS DOUBLE),
        CAST(total_amount_paid_per_bill AS DOUBLE), CAST(date_insurer_received_bill AS DATE),
        CAST(date_insurer_paid_bill AS DATE), CAST(employee_date_of_injury AS DATE),
        employee_gender_code, first_icd_9cm_or_icd_10cm, second_icd_9cm_or_icd_10cm
    FROM raw.institutional_header_historical
""").df()

print(f"Total bills loaded: {len(all_headers):,}")
print(f"Date range: {all_headers['bill_date'].min()} to {all_headers['bill_date'].max()}")
all_headers.groupby('claim_category').agg(
    bills=('bill_id','count'),
    total_charged=('charge','sum'),
    total_paid=('paid','sum')
).style.format({'total_charged': '${:,.0f}', 'total_paid': '${:,.0f}'})

Total bills loaded: 2,294
Date range: 2010-01-13 00:00:00 to 2026-01-22 00:00:00


,bills,total_charged,total_paid
claim_category,,,
institutional,610,"$8,858,814","$1,149,263"
pharmacy,782,"$136,404","$90,285"
professional,902,"$881,066","$257,358"


## 1. Overall Cost Trends by Claim Type

Annual total charges and payments across Professional, Institutional, and Pharmacy claims.

In [2]:
# Annual totals by claim type
annual = all_headers.copy()
annual['year'] = annual['bill_date'].dt.year
# Exclude partial 2026
annual = annual[annual['year'] <= 2025]

yearly = annual.groupby(['year', 'claim_category']).agg(
    n_bills=('bill_id', 'count'),
    total_charge=('charge', 'sum'),
    total_paid=('paid', 'sum'),
    avg_charge=('charge', 'mean'),
    avg_paid=('paid', 'mean'),
).reset_index()

# Stacked bar: total charges by claim type per year
fig = px.bar(yearly, x='year', y='total_charge', color='claim_category',
             title='Total Charges by Year and Claim Type',
             labels={'total_charge': 'Total Charges ($)', 'year': 'Year', 'claim_category': 'Claim Type'},
             color_discrete_map={'professional': '#636EFA', 'institutional': '#EF553B', 'pharmacy': '#00CC96'},
             barmode='stack')
fig.update_layout(xaxis=dict(dtick=1), height=450)
fig.show()

In [3]:
# Bill volume trend — line chart
fig = px.line(yearly, x='year', y='n_bills', color='claim_category',
              title='Number of Bills by Year and Claim Type',
              labels={'n_bills': 'Number of Bills', 'year': 'Year', 'claim_category': 'Claim Type'},
              markers=True,
              color_discrete_map={'professional': '#636EFA', 'institutional': '#EF553B', 'pharmacy': '#00CC96'})
fig.update_layout(xaxis=dict(dtick=1), height=400)
fig.show()

In [4]:
# Average charge per bill — trend over time
fig = px.line(yearly, x='year', y='avg_charge', color='claim_category',
              title='Average Charge per Bill by Year',
              labels={'avg_charge': 'Avg Charge ($)', 'year': 'Year', 'claim_category': 'Claim Type'},
              markers=True,
              color_discrete_map={'professional': '#636EFA', 'institutional': '#EF553B', 'pharmacy': '#00CC96'})
fig.update_layout(xaxis=dict(dtick=1), height=400)
fig.show()

## 2. Professional Claims (SV1) — Procedures & Diagnoses

Top billed CPT/HCPCS procedures, primary diagnosis codes, and how the procedure mix shifts over time.

In [5]:
# Professional detail — procedures with charges
prof_detail = con.sql("""
    SELECT
        bill_id,
        hcpcs_line_procedure_billed AS cpt_code,
        procedure_description AS proc_desc,
        CAST(total_charge_per_line AS DOUBLE) AS line_charge,
        CAST(total_amount_paid_per_line AS DOUBLE) AS line_paid,
        CAST(service_line_from_date AS DATE) AS service_date,
        CAST(days_units_billed AS DOUBLE) AS units_billed,
        CAST(days_units_paid AS DOUBLE) AS units_paid
    FROM (
        SELECT * FROM raw.professional_detail_current
        UNION ALL SELECT * FROM raw.professional_detail_historical
    )
    WHERE hcpcs_line_procedure_billed IS NOT NULL
""").df()

prof_detail['year'] = prof_detail['service_date'].dt.year

# Top 15 procedures by frequency
top_procs = prof_detail.groupby('cpt_code').agg(
    count=('cpt_code', 'size'),
    total_charge=('line_charge', 'sum'),
    total_paid=('line_paid', 'sum'),
    avg_charge=('line_charge', 'mean'),
).reset_index().sort_values('count', ascending=False).head(15)

# HCPCS descriptions lookup
hcpcs_desc = {
    '99080': 'Special reports/forms',
    '97110': 'Therapeutic exercises',
    '99213': 'Office visit (est, low)',
    '97530': 'Therapeutic activities',
    '97112': 'Neuromuscular re-ed',
    '97140': 'Manual therapy',
    '99214': 'Office visit (est, mod)',
    '99203': 'Office visit (new, low)',
    'G0283': 'Electrical stimulation',
    '99204': 'Office visit (new, mod)',
    '99456': 'Work disability exam',
    '97150': 'Group therapy',
    '99212': 'Office visit (est, brief)',
    '97546': 'Laser therapy',
    '97545': 'Work hardening',
}
top_procs['description'] = top_procs['cpt_code'].map(hcpcs_desc).fillna('')
top_procs['label'] = top_procs['cpt_code'] + ' — ' + top_procs['description']

fig = px.bar(top_procs, x='count', y='label', orientation='h',
             title='Top 15 Professional Procedures by Frequency',
             labels={'count': 'Line Count', 'label': ''},
             color='total_charge',
             color_continuous_scale='Blues',
             text='count')
fig.update_layout(yaxis=dict(autorange='reversed'), height=500, coloraxis_colorbar_title='Total $')
fig.show()

In [6]:
# Top 5 procedures over time — heatmap of annual volume
top5_codes = top_procs.head(5)['cpt_code'].tolist()
top5_detail = prof_detail[prof_detail['cpt_code'].isin(top5_codes) & (prof_detail['year'] <= 2025)]
proc_by_year = top5_detail.groupby(['year', 'cpt_code']).size().reset_index(name='count')

fig = px.line(proc_by_year, x='year', y='count', color='cpt_code', markers=True,
              title='Top 5 Professional Procedures — Annual Volume Trend',
              labels={'count': 'Line Count', 'year': 'Year', 'cpt_code': 'CPT/HCPCS'})
fig.update_layout(xaxis=dict(dtick=1), height=400)
fig.show()

In [7]:
# Primary diagnosis codes — top 15 across professional claims
prof_headers = all_headers[all_headers['claim_category'] == 'professional'].copy()
dx_counts = prof_headers['dx1'].value_counts().head(15).reset_index()
dx_counts.columns = ['icd_code', 'count']

# Classify ICD version
dx_counts['version'] = dx_counts['icd_code'].apply(lambda x: 'ICD-10' if '.' in str(x) and str(x)[0].isalpha() else 'ICD-9')

fig = px.bar(dx_counts, x='count', y='icd_code', orientation='h',
             color='version', title='Top 15 Primary Diagnosis Codes (Professional Claims)',
             labels={'count': 'Bill Count', 'icd_code': 'ICD Code'},
             text='count')
fig.update_layout(yaxis=dict(autorange='reversed'), height=500)
fig.show()

In [8]:
# ICD-10 diagnosis chapter grouping — what body systems are injured?
icd10_dx = prof_headers[prof_headers['dx1'].str.match(r'^[A-Z]', na=False)].copy()
icd10_dx['chapter'] = icd10_dx['dx1'].str[0]

chapter_map = {
    'S': 'Injury/Trauma (S)',
    'M': 'Musculoskeletal (M)',
    'T': 'Poisoning/External (T)',
    'G': 'Nervous System (G)',
    'R': 'Symptoms/Signs (R)',
    'H': 'Eye/Ear (H)',
    'K': 'Digestive (K)',
    'J': 'Respiratory (J)',
    'L': 'Skin (L)',
}
icd10_dx['chapter_label'] = icd10_dx['chapter'].map(chapter_map).fillna('Other')

chapter_counts = icd10_dx['chapter_label'].value_counts().reset_index()
chapter_counts.columns = ['chapter', 'count']

fig = px.pie(chapter_counts, values='count', names='chapter',
             title='ICD-10 Diagnosis Distribution by Body System (Professional Claims)',
             hole=0.3)
fig.update_layout(height=450)
fig.show()

## 3. Pharmacy Claims (SV4) — Drug Prescribing Trends

Top prescribed drugs, opioid vs. non-opioid patterns, and pharmacy cost trends over time.

In [9]:
# Pharmacy detail — drug-level data
rx_detail = con.sql("""
    SELECT
        bill_id,
        drug_name,
        ndc_billed_code AS ndc,
        CAST(drugs_supplies_billed_amount AS DOUBLE) AS billed_amount,
        CAST(total_amount_paid_per_line AS DOUBLE) AS paid_amount,
        CAST(drugs_supplies_quantity AS DOUBLE) AS quantity,
        CAST(drugs_supplies_number_of AS DOUBLE) AS days_supply,
        CAST(prescription_line_date AS DATE) AS rx_date,
        dispensed_as_written_code AS daw_code
    FROM (
        SELECT * FROM raw.pharmacy_detail_current
        UNION ALL SELECT * FROM raw.pharmacy_detail_historical
    )
    WHERE drug_name IS NOT NULL
""").df()

rx_detail['year'] = rx_detail['rx_date'].dt.year

# Top 15 drugs overall
top_drugs = rx_detail.groupby('drug_name').agg(
    rx_count=('drug_name', 'size'),
    total_billed=('billed_amount', 'sum'),
    total_paid=('paid_amount', 'sum'),
    avg_paid=('paid_amount', 'mean'),
    total_qty=('quantity', 'sum'),
).reset_index().sort_values('rx_count', ascending=False).head(15)

fig = px.bar(top_drugs, x='rx_count', y='drug_name', orientation='h',
             title='Top 15 Prescribed Drugs (Pharmacy Claims)',
             labels={'rx_count': 'Prescription Count', 'drug_name': ''},
             color='total_paid', color_continuous_scale='Greens',
             text='rx_count')
fig.update_layout(yaxis=dict(autorange='reversed'), height=500, coloraxis_colorbar_title='Total Paid $')
fig.show()

In [10]:
# Opioid vs Non-Opioid prescribing trend
opioid_keywords = ['HYDROCODONE', 'OXYCODONE', 'TRAMADOL', 'CODEINE', 'MORPHINE',
                   'FENTANYL', 'HYDROMORPHONE', 'METHADONE', 'BUPRENORPHINE', 'TAPENTADOL']

rx_detail['is_opioid'] = rx_detail['drug_name'].str.upper().apply(
    lambda x: any(k in str(x) for k in opioid_keywords)
)
rx_detail['drug_class'] = rx_detail['is_opioid'].map({True: 'Opioid', False: 'Non-Opioid'})

opioid_trend = rx_detail[rx_detail['year'].between(2010, 2025)].groupby(['year', 'drug_class']).agg(
    rx_count=('drug_name', 'size'),
    total_paid=('paid_amount', 'sum'),
).reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=['Prescription Count', 'Total Paid ($)'])

for i, metric in enumerate(['rx_count', 'total_paid'], 1):
    for cls in ['Opioid', 'Non-Opioid']:
        subset = opioid_trend[opioid_trend['drug_class'] == cls]
        fig.add_trace(go.Scatter(
            x=subset['year'], y=subset[metric], name=cls,
            mode='lines+markers', showlegend=(i == 1),
            line=dict(color='#EF553B' if cls == 'Opioid' else '#636EFA')
        ), row=1, col=i)

fig.update_layout(title='Opioid vs Non-Opioid Prescribing Trends', height=400,
                  xaxis=dict(dtick=2), xaxis2=dict(dtick=2))
fig.show()

# Opioid share
opioid_share = opioid_trend.pivot(index='year', columns='drug_class', values='rx_count').fillna(0)
opioid_share['opioid_pct'] = opioid_share['Opioid'] / (opioid_share['Opioid'] + opioid_share['Non-Opioid']) * 100
print("Opioid share of all pharmacy claims by year:")
print(opioid_share['opioid_pct'].round(1).to_string())

Opioid share of all pharmacy claims by year:
year
2010    23.7
2011    27.3
2012    22.2
2013    30.3
2014    31.7
2015    27.8
2016    27.8
2017    41.9
2018    51.4
2019    50.0
2020    31.0
2021    28.1
2022    18.2
2023     6.9
2024    14.3
2025     3.3


In [11]:
# Top 5 drugs over time
top5_drugs = top_drugs.head(5)['drug_name'].tolist()
rx_top5_trend = rx_detail[rx_detail['drug_name'].isin(top5_drugs) & rx_detail['year'].between(2010, 2025)]
rx_top5_by_year = rx_top5_trend.groupby(['year', 'drug_name']).size().reset_index(name='count')

fig = px.line(rx_top5_by_year, x='year', y='count', color='drug_name', markers=True,
              title='Top 5 Drugs — Annual Prescription Volume',
              labels={'count': 'Prescriptions', 'year': 'Year', 'drug_name': 'Drug'})
fig.update_layout(xaxis=dict(dtick=2), height=400)
fig.show()

## 4. Institutional Claims (SV2) — Facility Billing Patterns

Institutional claims tend to be the highest-cost category. This section examines charge distributions and outlier episodes.

In [12]:
# Institutional claims — charge distribution
inst = all_headers[all_headers['claim_category'] == 'institutional'].copy()
inst['year'] = inst['bill_date'].dt.year
inst = inst[inst['year'].between(2010, 2025)]

fig = px.box(inst, x='year', y='charge',
             title='Institutional Claim Charges by Year (Box Plot)',
             labels={'charge': 'Charge ($)', 'year': 'Year'})
fig.update_layout(xaxis=dict(dtick=1), height=450)
fig.update_yaxes(type='log', title='Charge ($ — log scale)')
fig.show()

# Summary stats
inst_yearly = inst.groupby('year').agg(
    n_bills=('bill_id', 'count'),
    median_charge=('charge', 'median'),
    mean_charge=('charge', 'mean'),
    p95_charge=('charge', lambda x: x.quantile(0.95)),
    total_charge=('charge', 'sum'),
    total_paid=('paid', 'sum'),
).reset_index()

inst_yearly['reimbursement_rate'] = (inst_yearly['total_paid'] / inst_yearly['total_charge'] * 100).round(1)
inst_yearly.style.format({
    'median_charge': '${:,.0f}', 'mean_charge': '${:,.0f}',
    'p95_charge': '${:,.0f}', 'total_charge': '${:,.0f}',
    'total_paid': '${:,.0f}', 'reimbursement_rate': '{:.1f}%'
})

,year,n_bills,median_charge,mean_charge,p95_charge,total_charge,total_paid,reimbursement_rate
0,2010,14,$805,"$1,755","$5,076","$24,565","$4,323",17.6%
1,2011,33,"$1,796","$2,222","$6,468","$73,340","$20,646",28.2%
2,2012,24,"$2,782","$7,736","$36,680","$185,660","$35,032",18.9%
3,2013,34,"$1,464","$10,451","$57,834","$355,341","$73,697",20.7%
4,2014,29,"$2,891","$7,697","$45,472","$223,201","$26,453",11.9%
5,2015,28,"$1,924","$16,134","$51,134","$451,759","$65,891",14.6%
6,2016,33,"$1,348","$5,382","$27,265","$177,609","$54,861",30.9%
7,2017,46,"$1,102","$10,630","$53,138","$488,966","$63,727",13.0%
8,2018,23,$920,"$2,425","$10,321","$55,765","$9,646",17.3%
9,2019,28,"$1,447","$4,997","$19,908","$139,914","$33,815",24.2%


## 5. Reimbursement Analysis — Paid-to-Billed Ratios & Payment Lag

How much of what's billed actually gets paid, and how long does it take?

In [13]:
# Reimbursement rate by claim type over time
reimb = all_headers.copy()
reimb['year'] = reimb['bill_date'].dt.year
reimb = reimb[reimb['year'].between(2010, 2025) & (reimb['charge'] > 0)]

reimb_trend = reimb.groupby(['year', 'claim_category']).agg(
    total_charge=('charge', 'sum'),
    total_paid=('paid', 'sum'),
).reset_index()
reimb_trend['reimb_rate'] = (reimb_trend['total_paid'] / reimb_trend['total_charge'] * 100)

fig = px.line(reimb_trend, x='year', y='reimb_rate', color='claim_category',
              title='Reimbursement Rate (Paid / Billed) by Year and Claim Type',
              labels={'reimb_rate': 'Reimbursement Rate (%)', 'year': 'Year', 'claim_category': 'Claim Type'},
              markers=True,
              color_discrete_map={'professional': '#636EFA', 'institutional': '#EF553B', 'pharmacy': '#00CC96'})
fig.update_layout(xaxis=dict(dtick=1), height=400)
fig.add_hline(y=100, line_dash='dash', line_color='gray', annotation_text='100% (full reimbursement)')
fig.show()

In [14]:
# Payment lag — days between bill date and insurer payment
lag = all_headers.dropna(subset=['bill_date', 'insurer_paid']).copy()
lag['pay_lag_days'] = (lag['insurer_paid'] - lag['bill_date']).dt.days
lag = lag[(lag['pay_lag_days'] >= 0) & (lag['pay_lag_days'] < 365)]  # filter outliers
lag['year'] = lag['bill_date'].dt.year
lag = lag[lag['year'].between(2010, 2025)]

lag_trend = lag.groupby(['year', 'claim_category'])['pay_lag_days'].agg(['mean', 'median']).reset_index()

fig = px.line(lag_trend, x='year', y='median', color='claim_category',
              title='Median Payment Lag (Days from Bill to Insurer Payment)',
              labels={'median': 'Median Days', 'year': 'Year', 'claim_category': 'Claim Type'},
              markers=True,
              color_discrete_map={'professional': '#636EFA', 'institutional': '#EF553B', 'pharmacy': '#00CC96'})
fig.update_layout(xaxis=dict(dtick=1), height=400)
fig.show()

## 6. Gender & Injury-to-Treatment Analysis

Demographics of the injured worker population and time from injury to first bill.

In [15]:
# Gender breakdown by claim type
gender_data = all_headers[all_headers['gender'].isin(['M', 'F'])].copy()
gender_data['year'] = gender_data['bill_date'].dt.year
gender_data = gender_data[gender_data['year'].between(2010, 2025)]

gender_by_year = gender_data.groupby(['year', 'gender']).size().reset_index(name='count')

fig = px.bar(gender_by_year, x='year', y='count', color='gender', barmode='group',
             title='Bill Volume by Gender Over Time',
             labels={'count': 'Number of Bills', 'year': 'Year', 'gender': 'Gender'},
             color_discrete_map={'M': '#636EFA', 'F': '#EF553B'})
fig.update_layout(xaxis=dict(dtick=1), height=400)
fig.show()

In [16]:
# Average charge by gender and claim type
gender_charge = gender_data.groupby(['claim_category', 'gender']).agg(
    avg_charge=('charge', 'mean'),
    avg_paid=('paid', 'mean'),
    n_bills=('bill_id', 'count')
).reset_index()

fig = px.bar(gender_charge, x='claim_category', y='avg_charge', color='gender', barmode='group',
             title='Average Charge per Bill by Gender and Claim Type',
             labels={'avg_charge': 'Avg Charge ($)', 'claim_category': 'Claim Type', 'gender': 'Gender'},
             text=gender_charge['avg_charge'].round(0).astype(int),
             color_discrete_map={'M': '#636EFA', 'F': '#EF553B'})
fig.update_layout(height=400)
fig.show()

In [17]:
# Time from injury to first bill
itb = all_headers.dropna(subset=['injury_date', 'bill_date']).copy()
itb['injury_to_bill_days'] = (itb['bill_date'] - itb['injury_date']).dt.days
itb = itb[(itb['injury_to_bill_days'] >= 0) & (itb['injury_to_bill_days'] <= 3650)]  # within 10 years

fig = px.histogram(itb, x='injury_to_bill_days', color='claim_category', nbins=60,
                   title='Distribution of Days from Injury to Bill Date',
                   labels={'injury_to_bill_days': 'Days from Injury to Bill', 'claim_category': 'Claim Type'},
                   color_discrete_map={'professional': '#636EFA', 'institutional': '#EF553B', 'pharmacy': '#00CC96'},
                   barmode='overlay', opacity=0.6)
fig.update_layout(height=400)
fig.show()

# Summary stats
print("Median days from injury to bill by claim type:")
print(itb.groupby('claim_category')['injury_to_bill_days'].median().round(0).to_string())

Median days from injury to bill by claim type:
claim_category
institutional    100.0
pharmacy         180.0
professional     105.0


## 7. Summary Dashboard — Key Metrics at a Glance

In [18]:
# Summary dashboard — 4 KPI panels
total_bills = len(all_headers)
total_charged = all_headers['charge'].sum()
total_paid = all_headers['paid'].sum()
overall_reimb = total_paid / total_charged * 100
opioid_total = rx_detail['is_opioid'].sum()
opioid_pct = opioid_total / len(rx_detail) * 100

fig = make_subplots(rows=2, cols=3, subplot_titles=[
    'Total Bills', 'Total Charged', 'Total Paid',
    'Reimbursement Rate', 'Opioid Rx Share', 'Unique Persons (OMOP)'
], specs=[[{'type': 'indicator'}]*3]*2)

fig.add_trace(go.Indicator(mode='number', value=total_bills, number=dict(font_size=40)), row=1, col=1)
fig.add_trace(go.Indicator(mode='number', value=total_charged, number=dict(prefix='$', font_size=40, valueformat=',.0f')), row=1, col=2)
fig.add_trace(go.Indicator(mode='number', value=total_paid, number=dict(prefix='$', font_size=40, valueformat=',.0f')), row=1, col=3)
fig.add_trace(go.Indicator(mode='number', value=overall_reimb, number=dict(suffix='%', font_size=40, valueformat='.1f')), row=2, col=1)
fig.add_trace(go.Indicator(mode='number', value=opioid_pct, number=dict(suffix='%', font_size=40, valueformat='.1f')), row=2, col=2)
fig.add_trace(go.Indicator(mode='number', value=1675, number=dict(font_size=40)), row=2, col=3)

fig.update_layout(height=350, title='Key Metrics — TX Workers Comp Claims (2010–2025)')
fig.show()

In [19]:
con.close()